## Mistral-7B-Instruct-v0.3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ========================== CELL 1: INSTALL & LOGIN ==========================
# CELL 1: Install dependencies & Hugging Face login
!pip install -q transformers accelerate bitsandbytes torch pandas tqdm

from huggingface_hub import login
login()  # ← paste your Hugging Face read token (Mistral models are public)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import pandas as pd
from tqdm import tqdm
from google.colab import files



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [3]:
# ========================== CELL 2: LOAD Mistral-7B-Instruct-v0.3 (4-bit) ==========================
# CELL 2: Load Mistral-7B-Instruct-v0.3 (4-bit)
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
# ========================== CELL 3: DEFINE 3 LLM AGENTS ==========================
# CELL 3: Define query function + 3 agents
def query_agent(system_prompt, user_message, max_new=180):
    # Mistral-Instruct chat format uses [INST] ... [/INST]
    full_prompt = f"[INST]{system_prompt}\n\n{user_message}[/INST]"

    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,                # greedy → more consistent JSON
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Mistral sometimes adds extra text → try to extract JSON part
    if "{" in response and "}" in response:
        start = response.find("{")
        end   = response.rfind("}") + 1
        return response[start:end]
    return response

# ─── Agent 1: Fact Checker ─────────────────────────────────────
fact_system = """You are FactCheckerAgent.
Given title, URL (may be empty), and review text (English or Traditional Chinese + emojis/links),
classify as:
- real : genuine product/service review
- fake(misinformation) : clearly false, spam, misleading
- unrelated : off-topic, not about the product

If title & URL are empty/null → judge only from text style/content.
Output ONLY this JSON, nothing else:
{"label": "real"} or {"label": "fake(misinformation)"} or {"label": "unrelated"}"""

# ─── Agent 2: Sentiment Analyzer ───────────────────────────────
sentiment_system = """You are SentimentAgent.
Analyze sentiment of the review text (English / Traditional Chinese / emojis / links ok).
Output ONLY this JSON, nothing else:
{"sentiment": "positive"} or {"sentiment": "negative"} or {"sentiment": "neutral"}"""

# ─── Agent 3: Combiner (final validator) ───────────────────────
combiner_system = """You are CombinerAgent — final judge.
Given:
- Title: ...
- URL: ...
- Text: ...
- Fact label: ...
- Sentiment: ...

Keep values unless obvious mistake, then correct.
Output ONLY JSON:
{"label": "...", "sentiment": "..."}"""


In [5]:
# ========================== CELL 4: LOAD DATA & RUN 3 AGENTS ==========================
# CELL 4: Load data — first 100 rows for testing

data_path = "/content/drive/MyDrive/Czech/M12_final_v1_to_dataset.jsonl"

reviews = []
with open(data_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.strip():
            try:
                reviews.append(json.loads(line))
            except:
                pass
        if len(reviews) >= 100:
            break

print(f"Loaded {len(reviews)} reviews (capped at 100 for testing)")

results = []

for row in tqdm(reviews, desc="3-Agent Mistral-7B Processing"):
    package_id = row.get("package_id", "")
    review_id  = row.get("review_id",  "")
    cid        = row.get("cid",        "")
    text       = row.get("text",       "")
    url        = row.get("url")   if row.get("url")   else ""
    title      = row.get("title") if row.get("title") else ""

    if not text.strip():
        label = "unrelated"
        sentiment = "neutral"
    else:
        # Agent 1
        fact_user = f"Title: {title}\nURL: {url}\nText: {text}"
        fact_raw = query_agent(fact_system, fact_user)
        try:
            label = json.loads(fact_raw)["label"]
        except:
            label = "unrelated"

        # Agent 2
        sent_user = f"Text: {text}"
        sent_raw = query_agent(sentiment_system, sent_user)
        try:
            sentiment = json.loads(sent_raw)["sentiment"]
        except:
            sentiment = "neutral"

        # Agent 3 — final check
        combiner_user = f"""Title: {title}
URL: {url}
Text: {text}
Fact label: {label}
Sentiment: {sentiment}"""
        final_raw = query_agent(combiner_system, combiner_user, max_new=120)
        try:
            final = json.loads(final_raw)
            label      = final.get("label", label)
            sentiment  = final.get("sentiment", sentiment)
        except:
            pass

    results.append({
        "package_id": package_id,
        "review_id":  review_id,
        "cid":        cid,
        "url":        url,
        "title":      title,
        "text":       text,
        "label":      label,
        "sentiment":  sentiment
    })



Loaded 100 reviews (capped at 100 for testing)


3-Agent Mistral-7B Processing: 100%|██████████| 100/100 [10:45<00:00,  6.46s/it]


In [6]:
# ========================== CELL 5: SAVE CSV ==========================

# CELL 5: Save & Download CSV
df = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/Czech/mistral7b-analyzed_reviews.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Done! Saved {len(results)} rows → {output_path}")
print(df.head(10))

# Optional: download
files.download(output_path)




Done! Saved 100 rows → /content/drive/MyDrive/Czech/mistral7b-analyzed_reviews.csv
  package_id  review_id         cid url title  \
0         01   ptt_30_1   ptt_30_c1             
1         01   ptt_30_2   ptt_30_c2             
2         01   ptt_30_3   ptt_30_c3             
3         01   ptt_30_4   ptt_30_c4             
4         01   ptt_30_5   ptt_30_c5             
5         01   ptt_30_6   ptt_30_c6             
6         01   ptt_30_7   ptt_30_c7             
7         01   ptt_30_8   ptt_30_c8             
8         01   ptt_30_9   ptt_30_c9             
9         01  ptt_30_10  ptt_30_c10             

                                    text      label sentiment  
0               都說了，就別執著了。而且也沒意義，吃不了那麼高。  unrelated   neutral  
1          走PD3.2，可能只有17PRO能吃滿也說不定，當專用即可  unrelated   neutral  
2                 #1emPiow4 (MobileComm)  unrelated   neutral  
3                              typec懂的都懂  unrelated   neutral  
4  我上個月買Google Pixel Flex 67W雙C充電頭 支援AVS       real  posi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>